In [1]:
# transformers - langchain을 안쓸 때 사용하게 되어있는 라이브러리
# bitsandbytes - 양자화(quantization)하여 모델을 더 작게 만들어 메모리 사용량을 줄이는 라이브러리
%pip install -q langchain transformers langchain-huggingface langchain-community langchain-core langchain-text-splitters bitsandbytes docx2txt langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 허깅 페이스에서 모델 불러오기 돌릴 때 허깅페이스 토큰을 넣어줘야함(허깅페이스 엑세스 토큰 발급받아서)
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-4-mini-instruct",
    task="text-generation",
    pipeline_kwargs=dict(
        # max_new_tokens을 지정하지 않으면 답변이 짧게 나온다.
        # 그래서 테스트해보면서 적절히 수정
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
)

chat_model = ChatHuggingFace(llm=llm)

In [ ]:
# 양자화(모델을 작게 만들어서 추론시간을 빠르게 만드는 기술)를 안해서
# 모델이 크면 동작 속도가 많이 느려진다.
ai_message = chat_model.invoke("What is the capital of France?")

In [ ]:
from transformers import BitsAndBytesConfig

# 양자화 설정
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# 양자화된 모델 불러오기
quantized_llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-4-mini-instruct",
    task="text-generation",
    pipeline_kwargs=dict(
        # max_new_tokens을 지정하지 않으면 답변이 짧게 나온다.
        # 그래서 테스트해보면서 적절히 수정
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
    model_kwargs={"quantization_config": quantization_config},
)

In [ ]:
quantized_chat_model = ChatHuggingFace(llm=quantized_llm)

In [ ]:
quantized_ai_meesage = quantized_chat_model.invoke("What is the capital of France?")